# 02 — Patient split and genomic feature preprocessing

Patient-level 70/15/15 split, common-panel mutation/CNA matrices and a combined representation. Data-dependent filtering during model fitting remains training-only.

## 1. Imports and data loading

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = Path.cwd()
if not (ROOT / "data" / "processed").is_dir():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "processed"
PANEL_DIR = ROOT / "data" / "raw" / "panels"
RANDOM_STATE = 42

clinical = pd.read_csv(DATA_DIR / "clinical_manifest.csv")
mutations = pd.read_csv(DATA_DIR / "mutations_long.csv")
cna = pd.read_csv(DATA_DIR / "cna_long.csv")
panels = pd.read_csv(DATA_DIR / "gene_panel_matrix.csv")
assert clinical["PATIENT_ID"].is_unique and clinical["SAMPLE_ID"].is_unique
assert clinical["stage_binary"].isin([0, 1]).all()
print("Patients:", len(clinical))

## 2. Reproducible patient-level split

In [ ]:
patients = clinical[["PATIENT_ID", "stage_binary"]].drop_duplicates("PATIENT_ID")
train, rest = train_test_split(patients, test_size=0.30,
                               stratify=patients["stage_binary"], random_state=RANDOM_STATE)
validation, test = train_test_split(rest, test_size=0.50,
                                    stratify=rest["stage_binary"], random_state=RANDOM_STATE)
splits = pd.concat([train.assign(split="train"),
                    validation.assign(split="validation"),
                    test.assign(split="test")], ignore_index=True)
assert splits["PATIENT_ID"].is_unique
assert len(splits) == len(patients)
assert set(splits["PATIENT_ID"]) == set(patients["PATIENT_ID"])
splits.to_csv(DATA_DIR / "patient_splits.csv", index=False)
print(pd.crosstab(splits["split"], splits["stage_binary"], margins=True))

## 3. Copy-number feature matrix

Preserve original CNA categories; subset shared-panel genes after the gene-panel intersection has been established.

In [ ]:
assert not cna.duplicated(["SAMPLE_ID", "HUGO_SYMBOL"]).any()
X_cna = cna.pivot(index="SAMPLE_ID", columns="HUGO_SYMBOL", values="CNA_VALUE")
X_cna = X_cna.reindex(clinical["SAMPLE_ID"].tolist())
X_cna.columns = [f"CNA__{gene}" for gene in X_cna.columns]
X_cna.index = pd.Index(clinical["PATIENT_ID"], name="PATIENT_ID")
assert X_cna.index.is_unique and not X_cna.isna().any().any()
X_cna.to_csv(DATA_DIR / "cna_feature_matrix.csv")
print("Unfiltered CNA matrix:", X_cna.shape)

## 4. Panel definitions and mutation coverage audit

Gene-list mismatch does not imply that the original variant is false. In this course analysis, unmatched records are excluded from feature construction and the exclusion count is documented.

In [ ]:
panel_ids = ["IMPACT341", "IMPACT410", "IMPACT468", "IMPACT505"]
panel_genes = {}
for panel_id in panel_ids:
    with open(PANEL_DIR / f"{panel_id}.json", encoding="utf-8-sig") as f:
        panel = json.load(f)
    panel_genes[panel_id] = {g["hugoGeneSymbol"] for g in panel["genes"]}

sample_panels = panels[["SAMPLE_ID", "mutations"]].rename(columns={"mutations": "PANEL_ID"})
audit = mutations.merge(sample_panels, on="SAMPLE_ID", how="left", validate="many_to_one")
assert audit["PANEL_ID"].isin(panel_genes).all()
audit["covered"] = [gene in panel_genes[pid]
                    for gene, pid in zip(audit["HUGO_SYMBOL"], audit["PANEL_ID"])]
print(f"Mutations total: {len(audit):,}; matched: {audit['covered'].sum():,}; "
      f"excluded: {(~audit['covered']).sum():,}")

## 5. Common-panel mutation feature matrix

Use genes shared by all four panels, encoding a reported mutation as `1` and no reported mutation in a covered gene as `0`.

In [ ]:
common_genes = sorted(set.intersection(*panel_genes.values()))
X_mut = pd.DataFrame(0, index=pd.Index(clinical["PATIENT_ID"], name="PATIENT_ID"),
                     columns=[f"MUT__{g}" for g in common_genes], dtype="int8")
valid_mutations = audit.loc[audit["covered"], ["PATIENT_ID", "HUGO_SYMBOL"]].drop_duplicates()
common_set = set(common_genes)
for patient_id, gene in valid_mutations.itertuples(index=False, name=None):
    if gene in common_set:
        X_mut.at[patient_id, f"MUT__{gene}"] = 1
assert X_mut.index.equals(X_cna.index)
assert X_mut.isin([0, 1]).all().all()
X_mut.to_csv(DATA_DIR / "mutation_feature_matrix.csv")
print("Mutation matrix:", X_mut.shape)

## 6. Shared-panel CNA and combined matrices

In [ ]:
available_cna_genes = {c.removeprefix("CNA__") for c in X_cna.columns}
common_cna = sorted(common_set & available_cna_genes)
X_cna_common = X_cna[[f"CNA__{g}" for g in common_cna]].copy()
X_combined = pd.concat([X_mut, X_cna_common], axis=1)
assert X_combined.index.is_unique and X_combined.columns.is_unique
assert X_combined.index.equals(X_mut.index)
assert not X_combined.isna().any().any()
X_cna_common.to_csv(DATA_DIR / "cna_feature_matrix_common.csv")
X_combined.to_csv(DATA_DIR / "combined_feature_matrix.csv")
print("Mutation:", X_mut.shape)
print("CNA:", X_cna_common.shape)
print("Combined:", X_combined.shape)
print("PASS: Feature matrices saved for notebook 03")

## 7. Notes

Only the pre-specified panel intersection is applied here. Constant-feature removal, standardization and hyperparameter selection are performed **inside training-only pipelines** in notebook 03. `STAGE_HIGHEST_RECORDED` should not be interpreted as stage at biopsy or future progression.